# Аннотация IgBLAST — мышь `ERP003950`

Production-этап аннотации для ветки fastp Q30/u40.

Notebook выполняет exact-collapse объединённых последовательностей, запускает IgBLAST
для уникальных nucleotide sequences и восстанавливает annotation для каждого исходного
merged read. При наличии полного совместимого cache результаты exact-collapse и IgBLAST
переиспользуются только после проверки:

- collapsed FASTA и AIRR существуют;
- число FASTA records совпадает с числом AIRR rows;
- множества `sequence_id` совпадают;
- поле AIRR `sequence` соответствует последовательности collapsed FASTA.

Если cache не проходит проверку, exact-collapse и IgBLAST выполняются заново.

Выходные данные:

```text
results/ERP003950/annotation/
├── unique/
│   ├── {sample}.collapse-unique.fasta
│   └── {sample}.unique.airr.tsv
├── igblast/
│   ├── {sample}.airr.tsv
│   └── {sample}.manifest.json
├── index/
├── logs/
└── annotation_summary.json
```

`annotation/igblast/{sample}.airr.tsv` — основной downstream input.


In [ ]:
import csv, gzip, json, os, shutil, sqlite3, subprocess, sys, time
from pathlib import Path

RESULT_DATASET = "ERP003950"
SAMPLES = [
    "ERR346596", "ERR346597", "ERR346598",
    "ERR346599", "ERR346600", "ERR346601",
]

NPROC = 4
FORCE = False
KEEP_ASSEMBLED_FASTA = False

def resolve_volume():
    candidates = []
    if os.environ.get("BCR_VOLUME"):
        candidates.append(Path(os.environ["BCR_VOLUME"]))
    candidates += [
        Path("/data/user/epishkin"),
        Path("/Users/epishkin/workspace/bcr-assembler"),
    ]
    start = Path.cwd().resolve()
    candidates += [start, *start.parents]

    seen = set()
    for root in candidates:
        if str(root) in seen:
            continue
        seen.add(str(root))
        if (root / "results" / RESULT_DATASET / "merged" / "fastq").is_dir():
            return root

    raise FileNotFoundError(
        f"Cannot locate results/{RESULT_DATASET}/merged/fastq. "
        "Set BCR_VOLUME explicitly."
    )

VOLUME = resolve_volume()
DATASET_DIR = VOLUME / "results" / RESULT_DATASET
MERGED_FASTQ_DIR = DATASET_DIR / "merged" / "fastq"
LEGACY_COMPARE_DIR = DATASET_DIR / "comparison_annotation"

ANNOTATION_DIR = DATASET_DIR / "annotation"
UNIQUE_DIR = ANNOTATION_DIR / "unique"
IGBLAST_DIR = ANNOTATION_DIR / "igblast"
INDEX_DIR = ANNOTATION_DIR / "index"
LOG_DIR = ANNOTATION_DIR / "logs"

for d in (UNIQUE_DIR, IGBLAST_DIR, INDEX_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Определяем доступные layouts IgBLAST для текущего окружения.
ENV_CANDIDATES = [
    os.environ.get("BCR_ENV", ""),
    os.environ.get("CONDA_PREFIX", ""),
    "/data/user/epishkin/conda/envs/bcr_env",
    "/opt/conda/envs/bcr_env",
    "/Users/epishkin/mamba/envs/bcr_env",
]
BCR_ENV = next(
    (Path(p) for p in ENV_CANDIDATES if p and (Path(p) / "bin").is_dir()),
    None,
)
if BCR_ENV is None:
    raise FileNotFoundError("Cannot find bcr_env; activate it or set BCR_ENV")

igdata_candidates = [
    Path(os.environ["IGDATA"]) if os.environ.get("IGDATA") else None,
    Path("/data/user/epishkin/igblast"),
    BCR_ENV / "share" / "igblast",
]
IGBLASTN = next(
    (root / "bin" / "igblastn" for root in igdata_candidates
     if root is not None and (root / "bin" / "igblastn").exists()),
    BCR_ENV / "bin" / "igblastn",
)

db_candidates = [
    (root / "internal_data" / "mouse") for root in igdata_candidates
    if root is not None
] + [
    VOLUME / "results/PRJNA1226555/references/germline/ncbi_mouse",
]
DB = next(
    (root for root in db_candidates if (root / "mouse_gl_V.nhr").exists()),
    None,
)
if DB is None:
    raise FileNotFoundError(f"Cannot find mouse IgBLAST DB in {db_candidates}")

aux_candidates = [
    (root / "optional_file" / "mouse_gl.aux") for root in igdata_candidates
    if root is not None
] + [
    VOLUME / "results/PRJNA1226555/references/germline/igdata_balbcbyj/optional_file/mouse_gl.aux",
]
AUX = next((path for path in aux_candidates if path.exists()), None)
if AUX is None:
    raise FileNotFoundError(f"Cannot find mouse_gl.aux in {aux_candidates}")

GERMLINE_V = DB / "mouse_gl_V"
GERMLINE_D = DB / "mouse_gl_D"
GERMLINE_J = DB / "mouse_gl_J"
IGDATA = AUX.parents[1]
COLLAPSESEQ = BCR_ENV / "bin" / "CollapseSeq.py"

for name, path in {
    "igblastn": IGBLASTN,
    "CollapseSeq.py": COLLAPSESEQ,
    "V DB": GERMLINE_V.with_suffix(".nhr"),
    "D DB": GERMLINE_D.with_suffix(".nhr"),
    "J DB": GERMLINE_J.with_suffix(".nhr"),
    "AUX": AUX,
}.items():
    if not path.exists():
        raise FileNotFoundError(f"{name}: {path}")

found = sorted(
    p.name.removesuffix("_assemble-pass.fastq.gz")
    for p in MERGED_FASTQ_DIR.glob("*_assemble-pass.fastq.gz")
)
if found != SAMPLES:
    raise RuntimeError(f"Unexpected sample set: {found}")

print("VOLUME:", VOLUME)
print("DATASET_DIR:", DATASET_DIR)
print("LEGACY_COMPARE_DIR:", LEGACY_COMPARE_DIR)
print("IGBLASTN:", IGBLASTN)
print("CollapseSeq.py:", COLLAPSESEQ)
print("samples:", SAMPLES)


In [ ]:
def run_with_heartbeat(cmd, stdout_path, stderr_path, heartbeat=60, env=None):
    stdout_path = Path(stdout_path)
    stderr_path = Path(stderr_path)
    t0 = time.time()

    with stdout_path.open("w") as out, stderr_path.open("w") as err:
        proc = subprocess.Popen(
            [str(x) for x in cmd],
            stdout=out,
            stderr=err,
            text=True,
            env=env,
        )
        print("PID", proc.pid, " ".join(map(str, cmd[:3])), flush=True)

        while proc.poll() is None:
            print(f"  elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(heartbeat)

    if proc.returncode != 0:
        raise RuntimeError(
            f"Command failed rc={proc.returncode}; inspect {stderr_path}"
        )

def iter_fastq(path):
    with gzip.open(path, "rt") as h:
        while True:
            head = h.readline()
            if not head:
                return
            seq = h.readline()
            plus = h.readline()
            qual = h.readline()
            if not qual or not head.startswith("@") or not plus.startswith("+"):
                raise ValueError(f"Malformed FASTQ: {path}")
            yield head, seq, plus, qual

def fastq_id(header):
    return header[1:].strip().split(None, 1)[0]

_COMPLEMENT = str.maketrans(
    "ACGTRYKMSWBDHVN",
    "TGCAYRMKSWVHDBN",
)

def reverse_complement(sequence):
    return str(sequence).upper().translate(_COMPLEMENT)[::-1]

def has_ambiguous_bases(sequence):
    return any(base not in {"A", "C", "G", "T"} for base in str(sequence).upper())

def is_reverse_complemented(row):
    return str(row.get("rev_comp", "")).strip().lower() in {
        "t", "true", "1", "y", "yes",
    }

def expected_airr_sequence(query_sequence, row):
    query_sequence = str(query_sequence).upper()
    return (
        reverse_complement(query_sequence)
        if is_reverse_complemented(row)
        else query_sequence
    )

def iter_fasta(path):
    with open(path) as h:
        name = None
        chunks = []
        for line in h:
            line = line.rstrip("\r\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks).upper()
                name = line[1:].split()[0]
                chunks = []
            else:
                chunks.append(line)
        if name is not None:
            yield name, "".join(chunks).upper()

def fasta_count(path):
    return sum(1 for _ in iter_fasta(path))

def tsv_count(path):
    with open(path, errors="replace") as h:
        return max(0, sum(1 for _ in h) - 1)

def fastq_to_fasta(fastq, fasta):
    tmp = Path(str(fasta) + ".tmp")
    n = 0
    with tmp.open("w") as out:
        for head, seq, _, _ in iter_fastq(fastq):
            out.write(f">{fastq_id(head)}\n{seq.strip().upper()}\n")
            n += 1
    tmp.replace(fasta)
    return n

def source_signature(path):
    st = Path(path).stat()
    return {"size": st.st_size, "mtime_ns": st.st_mtime_ns}

def legacy_paths(sample):
    return (
        LEGACY_COMPARE_DIR / f"{sample}.collapse-unique.fasta",
        LEGACY_COMPARE_DIR / f"{sample}.mouse_igh.airr.tsv",
    )

def validate_legacy_cache(sample, verbose=True):
    """Return (ok, details) for legacy compare-heavy exact-collapse/IgBLAST cache."""
    fasta, airr = legacy_paths(sample)

    details = {
        "sample": sample,
        "collapsed_fasta": str(fasta),
        "airr": str(airr),
        "usable": False,
        "reason": None,
    }

    if not fasta.exists() or not airr.exists():
        details["reason"] = "legacy files missing"
        return False, details

    try:
        fasta_by_id = {}
        for sid, seq in iter_fasta(fasta):
            if sid in fasta_by_id:
                raise ValueError(f"duplicate FASTA id: {sid}")
            fasta_by_id[sid] = seq

        airr_ids = set()
        n_airr = 0

        with open(airr, newline="") as h:
            reader = csv.DictReader(h, delimiter="\t")
            fields = reader.fieldnames or []

            required = {"sequence_id", "sequence"}
            if not required.issubset(fields):
                raise ValueError(
                    f"AIRR missing fields: {sorted(required - set(fields))}"
                )

            for row in reader:
                sid = row["sequence_id"]
                if sid in airr_ids:
                    raise ValueError(f"duplicate AIRR sequence_id: {sid}")
                if sid not in fasta_by_id:
                    raise ValueError(f"AIRR id absent from collapsed FASTA: {sid}")

                airr_seq = (row.get("sequence") or "").upper()
                if not airr_seq:
                    raise ValueError(f"empty AIRR sequence for {sid}")
                expected_seq = expected_airr_sequence(fasta_by_id[sid], row)
                if airr_seq != expected_seq:
                    raise ValueError(f"sequence/orientation mismatch for {sid}")

                airr_ids.add(sid)
                n_airr += 1

        fasta_ids = set(fasta_by_id)
        if airr_ids != fasta_ids:
            missing = fasta_ids - airr_ids
            extra = airr_ids - fasta_ids
            raise ValueError(
                f"ID-set mismatch: missing={len(missing)} extra={len(extra)}"
            )

        details.update({
            "usable": True,
            "reason": "validated",
            "collapsed_fasta_records": len(fasta_by_id),
            "airr_rows": n_airr,
        })

        if verbose:
            print(
                f"[{sample}] validated legacy compare-heavy cache: "
                f"{n_airr:,} exact unique annotations"
            )

        return True, details

    except Exception as exc:
        details["reason"] = str(exc)
        if verbose:
            print(
                f"[{sample}] legacy cache rejected: {exc}; "
                "will fall back to production collapse/IgBLAST"
            )
        return False, details

def import_legacy_cache(sample, force=False):
    """Copy a validated legacy unique-query/AIRR pair into production annotation/unique."""
    ok, details = validate_legacy_cache(sample)
    if not ok:
        return False, details

    legacy_fasta, legacy_airr = legacy_paths(sample)
    prod_fasta = UNIQUE_DIR / f"{sample}.collapse-unique.fasta"
    prod_airr = UNIQUE_DIR / f"{sample}.unique.airr.tsv"

    if force or not prod_fasta.exists():
        shutil.copy2(legacy_fasta, prod_fasta)
    if force or not prod_airr.exists():
        shutil.copy2(legacy_airr, prod_airr)

    details["imported_to"] = {
        "collapsed_fasta": str(prod_fasta),
        "unique_airr": str(prod_airr),
    }
    return True, details


## 1. Exact-collapse объединённых последовательностей

IgBLAST получает только точные уникальные nucleotide sequences. Это вычислительная
оптимизация, а не биологическая дедупликация: исходная read multiplicity восстанавливается
в конце этапа аннотации.


In [ ]:
def build_unique_query(sample, force=FORCE):
    merged = MERGED_FASTQ_DIR / f"{sample}_assemble-pass.fastq.gz"
    assembled_fasta = UNIQUE_DIR / f"{sample}.assemble-pass.fasta"
    collapsed = UNIQUE_DIR / f"{sample}.collapse-unique.fasta"

    # При наличии полного cache импортируем exact-collapse и IgBLAST,
    # чтобы дальнейшие этапы не зависели от источника annotation.
    # Cache используется только после полной проверки согласованности.
    reused, details = import_legacy_cache(sample, force=force)
    if reused:
        print(
            f"[{sample}] [reuse] compare-heavy cache imported: "
            f"{details['airr_rows']:,} unique annotations"
        )
        return collapsed

    if collapsed.exists() and collapsed.stat().st_size > 0 and not force:
        print(
            f"[{sample}] [skip] production collapsed query exists: "
            f"{fasta_count(collapsed):,} unique"
        )
        return collapsed

    print(f"[{sample}] materializing merged FASTQ as FASTA")
    n_reads = fastq_to_fasta(merged, assembled_fasta)
    print(f"  merged reads: {n_reads:,}")

    tmp = UNIQUE_DIR / f"{sample}.collapse-unique.tmp.fasta"
    tmp.unlink(missing_ok=True)

    cmd = [
        COLLAPSESEQ,
        "-s", assembled_fasta,
        "-o", tmp,
        "--fasta",
        "-n", "0",
    ]
    run_with_heartbeat(
        cmd,
        LOG_DIR / f"{sample}.collapse.stdout.log",
        LOG_DIR / f"{sample}.collapse.stderr.log",
    )

    if not tmp.exists() or tmp.stat().st_size == 0:
        raise RuntimeError(f"{sample}: CollapseSeq produced no output")

    tmp.replace(collapsed)
    print(f"  exact unique: {fasta_count(collapsed):,}")

    if not KEEP_ASSEMBLED_FASTA:
        assembled_fasta.unlink(missing_ok=True)

    return collapsed


## 2. IgBLAST для уникальных последовательностей

Germline database и параметры mouse IgBLAST совпадают с canonical-аннотацией
`ERP003950`, поэтому preprocessing-ветки остаются сопоставимыми.


In [ ]:
def validate_production_unique_pair(sample):
    query = UNIQUE_DIR / f"{sample}.collapse-unique.fasta"
    airr = UNIQUE_DIR / f"{sample}.unique.airr.tsv"

    if not query.exists() or not airr.exists():
        return False

    fasta_by_id = dict(iter_fasta(query))
    airr_ids = set()

    with open(airr, newline="") as h:
        reader = csv.DictReader(h, delimiter="\t")
        fields = reader.fieldnames or []
        if not {"sequence_id", "sequence"}.issubset(fields):
            return False

        for row in reader:
            sid = row["sequence_id"]
            if sid in airr_ids or sid not in fasta_by_id:
                return False
            airr_seq = (row.get("sequence") or "").upper()
            if airr_seq != expected_airr_sequence(fasta_by_id[sid], row):
                return False
            airr_ids.add(sid)

    return airr_ids == set(fasta_by_id)

def run_unique_igblast(sample, force=FORCE):
    query = build_unique_query(sample, force=force)
    out = UNIQUE_DIR / f"{sample}.unique.airr.tsv"
    expected = fasta_count(query)

    # Проверка применяется как к существующему production output, так и к
    # импортированному совместимому cache.
    if out.exists() and not force and validate_production_unique_pair(sample):
        observed = tsv_count(out)
        print(
            f"[{sample}] [skip/reuse] unique AIRR validated: "
            f"{observed:,}/{expected:,}"
        )
        return out

    if out.exists() and not force:
        print(
            f"[{sample}] existing unique AIRR failed validation; rebuilding"
        )

    tmp = Path(str(out) + ".tmp")
    tmp.unlink(missing_ok=True)

    env = os.environ.copy()
    env["IGDATA"] = str(IGDATA)
    env["LD_LIBRARY_PATH"] = (
        str(BCR_ENV / "lib")
        + (":" + env["LD_LIBRARY_PATH"] if env.get("LD_LIBRARY_PATH") else "")
    )

    cmd = [
        IGBLASTN,
        "-query", query,
        "-organism", "mouse",
        "-ig_seqtype", "Ig",
        "-germline_db_V", GERMLINE_V,
        "-germline_db_D", GERMLINE_D,
        "-germline_db_J", GERMLINE_J,
        "-auxiliary_data", AUX,
        "-domain_system", "imgt",
        "-outfmt", "19",
        "-num_threads", str(NPROC),
        "-out", tmp,
    ]

    run_with_heartbeat(
        cmd,
        LOG_DIR / f"{sample}.igblast.stdout.log",
        LOG_DIR / f"{sample}.igblast.stderr.log",
        env=env,
        heartbeat=120,
    )

    observed = tsv_count(tmp)
    if observed != expected:
        raise RuntimeError(
            f"{sample}: unique AIRR row count {observed:,} != query count {expected:,}"
        )

    tmp.replace(out)

    if not validate_production_unique_pair(sample):
        raise RuntimeError(
            f"{sample}: newly generated unique AIRR failed exact query validation"
        )

    print(f"[{sample}] unique AIRR complete: {observed:,}")
    return out


## 3. Восстановление per-read AIRR

Disk-backed SQLite index связывает каждую уникальную nucleotide sequence с полной AIRR
строкой. Затем исходный merged FASTQ читается последовательно, и каждый read получает
annotation своей точной nucleotide sequence.

Из исходного read восстанавливаются только `sequence_id` и `sequence`; V/D/J calls,
coordinates, productivity, identities, supports и остальные поля IgBLAST не изменяются.

Такой подход не требует хранить сотни тысяч полных AIRR rows в RAM.


In [ ]:
def build_sqlite_index(sample, unique_airr, force=FORCE):
    db = INDEX_DIR / f"{sample}.unique_airr.sqlite"

    if db.exists() and force:
        db.unlink()

    if db.exists() and db.stat().st_size > 0:
        return db

    conn = sqlite3.connect(db)
    try:
        conn.execute("PRAGMA journal_mode=OFF")
        conn.execute("PRAGMA synchronous=OFF")
        conn.execute("PRAGMA temp_store=MEMORY")
        conn.execute(
            "CREATE TABLE ann (sequence TEXT PRIMARY KEY, row_json TEXT NOT NULL)"
        )

        n = 0
        with open(unique_airr, newline="") as h:
            reader = csv.DictReader(h, delimiter="\t")
            for row in reader:
                airr_sequence = (row.get("sequence") or "").upper()
                if not airr_sequence:
                    raise RuntimeError(
                        f"{sample}: AIRR row {row.get('sequence_id')} has empty sequence"
                    )
                query_sequence = (
                    reverse_complement(airr_sequence)
                    if is_reverse_complemented(row)
                    else airr_sequence
                )
                try:
                    conn.execute(
                        "INSERT INTO ann(sequence,row_json) VALUES (?,?)",
                        (query_sequence, json.dumps(row, separators=(",", ":"))),
                    )
                except sqlite3.IntegrityError as exc:
                    raise RuntimeError(
                        f"{sample}: duplicate exact sequence in unique AIRR"
                    ) from exc
                n += 1

        conn.commit()
        print(f"[{sample}] SQLite AIRR index: {n:,} unique sequences")
    finally:
        conn.close()

    return db

def project_to_per_read_airr(sample, force=FORCE):
    merged = MERGED_FASTQ_DIR / f"{sample}_assemble-pass.fastq.gz"
    unique_airr = run_unique_igblast(sample, force=force)

    out = IGBLAST_DIR / f"{sample}.airr.tsv"
    manifest = IGBLAST_DIR / f"{sample}.manifest.json"

    signature = source_signature(merged)

    if out.exists() and manifest.exists() and not force:
        old = json.loads(manifest.read_text())
        if old.get("merged_fastq_signature") == signature and old.get("complete") is True:
            print(
                f"[{sample}] [skip] per-read AIRR complete: "
                f"{old['per_read_airr_rows']:,}"
            )
            return old

    db = build_sqlite_index(sample, unique_airr, force=force)

    with open(unique_airr, newline="") as h:
        fields = csv.DictReader(h, delimiter="\t").fieldnames or []
    if "sequence_id" not in fields or "sequence" not in fields:
        raise RuntimeError(f"{unique_airr}: AIRR schema lacks sequence_id/sequence")

    tmp = Path(str(out) + ".tmp")
    tmp.unlink(missing_ok=True)

    conn = sqlite3.connect(db)
    conn.execute("PRAGMA query_only=ON")
    cursor = conn.cursor()

    n_reads = 0
    missing = 0
    unannotated_ambiguous_reads = 0

    try:
        with tmp.open("w", newline="") as oh:
            writer = csv.DictWriter(
                oh,
                fieldnames=fields,
                delimiter="\t",
                lineterminator="\n",
            )
            writer.writeheader()

            for head, seq_line, _, _ in iter_fastq(merged):
                sequence_id = fastq_id(head)
                sequence = seq_line.strip().upper()

                hit = cursor.execute(
                    "SELECT row_json FROM ann WHERE sequence=?",
                    (sequence,),
                ).fetchone()

                if hit is None:
                    if has_ambiguous_bases(sequence):
                        row = {field: "" for field in fields}
                        row["sequence_id"] = sequence_id
                        row["sequence"] = sequence
                        if "rev_comp" in row:
                            row["rev_comp"] = "F"
                        writer.writerow(row)
                        n_reads += 1
                        unannotated_ambiguous_reads += 1
                        continue
                    missing += 1
                    if missing <= 5:
                        print(
                            f"[{sample}] missing exact ACGT annotation: {sequence_id}",
                            file=sys.stderr,
                        )
                    continue

                row = json.loads(hit[0])
                row["sequence_id"] = sequence_id
                writer.writerow(row)
                n_reads += 1
    finally:
        conn.close()

    if missing:
        tmp.unlink(missing_ok=True)
        raise RuntimeError(
            f"{sample}: {missing:,} merged reads were absent from exact unique AIRR"
        )

    tmp.replace(out)

    info = {
        "dataset": RESULT_DATASET,
        "sample": sample,
        "method": "exact-collapse -> IgBLAST unique -> exact-sequence projection",
        "legacy_compare_cache_available": validate_legacy_cache(sample, verbose=False)[0],
        "merged_fastq": str(merged),
        "merged_fastq_signature": signature,
        "unique_query": str(UNIQUE_DIR / f"{sample}.collapse-unique.fasta"),
        "unique_airr": str(unique_airr),
        "unique_airr_rows": tsv_count(unique_airr),
        "per_read_airr": str(out),
        "per_read_airr_rows": n_reads,
        "unannotated_ambiguous_reads": unannotated_ambiguous_reads,
        "complete": True,
    }
    manifest.write_text(json.dumps(info, indent=2) + "\n")

    print(
        f"[{sample}] per-read AIRR: {n_reads:,}; "
        f"unique annotated: {info['unique_airr_rows']:,}; "
        f"ambiguous unannotated: {unannotated_ambiguous_reads:,}"
    )
    return info


In [ ]:
# Выполняем аннотацию для всех шести samples.
manifests = {}
legacy_cache = {}

for sample in SAMPLES:
    print("\n===", sample, "===")
    legacy_cache[sample] = validate_legacy_cache(sample, verbose=True)[1]
    manifests[sample] = project_to_per_read_airr(sample, force=FORCE)

summary = {
    "dataset": RESULT_DATASET,
    "annotation": "mouse IgBLAST outfmt 19",
    "optimization": (
        "exact-collapse before IgBLAST, exact-sequence projection back to reads"
    ),
    "legacy_compare_cache": legacy_cache,
    "samples": manifests,
    "total_per_read_airr_rows": sum(
        x["per_read_airr_rows"] for x in manifests.values()
    ),
    "total_unique_airr_rows": sum(
        x["unique_airr_rows"] for x in manifests.values()
    ),
}

(ANNOTATION_DIR / "annotation_summary.json").write_text(
    json.dumps(summary, indent=2) + "\n"
)

summary


## Результат

Этап считается завершённым, когда для всех шести samples manifests содержат `complete: true`.

Канонический downstream-вход:

`results/ERP003950/annotation/igblast/{sample}.airr.tsv`
